In [ ]:
!pip --quiet install sentence-transformers datasets plotly python-dotenv
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d mohammedaminejebbar/malicious-prompt-detection-dataset-mpdd
!unzip -o malicious-prompt-detection-dataset-mpdd.zip

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import torch
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from scipy.spatial import ConvexHull, Delaunay
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

random_seed = 345
embed_model = "BAAI/bge-large-en-v1.5"

def get_embeds(texts:list[str], model_name=embed_model, batch_size=256, cache_name='embeddings_cache.pkl'):
  if os.path.exists(cache_name):
    with open(cache_name, 'rb') as cache:
      return pickle.load(cache)

  print(f"Loading model {model_name} on GPU...")
  device = 'cuda' if torch.cuda.is_available() else 'cpu'
  model = SentenceTransformer(model_name, device=device)

  print("Calculating embeddings...")
  embeddings = model.encode(texts, batch_size=batch_size, show_progress_bar=True)

  embeds = np.array(embeddings)
  with open(cache_name, 'wb') as cache:
    pickle.dump(embeds, cache)

  return embeds

In [ ]:
print("Loading MPDD...")
df = pd.read_csv('MPDD.csv')

# Clean dataset (removing known anomalous rows from the original experiment)
LARGE_PROMPT_INDEX = [3545, 11324, 21042]
df = df.drop(index=LARGE_PROMPT_INDEX, errors='ignore')

texts = df['Prompt'].tolist()
labels = df['isMalicious'].values

# Generate Embeddings
embeddings = get_embeds(texts)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(embeddings, labels, test_size=0.2, random_state=random_seed)

# Dimensionality Reduction (PCA)
pca = PCA(n_components=3, random_state=random_seed)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)
print("PCA Transformation Complete!")

In [ ]:
class ClusterCHClassifier:
  def __init__(self, eps=0.5, min_samples=5):
    self.eps = eps
    self.min_samples = min_samples
    self.benign_hulls = []

  def fit(self, X, y):
    benign_points = X[y == 0]
    dbscan = DBSCAN(eps=self.eps, min_samples=self.min_samples)
    labels = dbscan.fit_predict(benign_points)
    self.benign_hulls = []
    unique_labels = set(labels)

    for label in unique_labels:
      if label == -1: continue
      cluster_points = benign_points[labels == label]
      if len(cluster_points) >= X.shape[1] + 1:
        try:
          hull = ConvexHull(cluster_points)
          delaunay = Delaunay(hull.points[hull.vertices])
          self.benign_hulls.append(delaunay)
        except Exception as e:
          pass

  def predict(self, X):
    predictions = np.ones(X.shape[0])
    if not self.benign_hulls: return predictions
    for delaunay in self.benign_hulls:
      is_inside = delaunay.find_simplex(X) >= 0
      predictions[is_inside] = 0
    return predictions


class NearestClusterCHClassifier:
  def __init__(self, eps=0.5, min_samples=5):
    self.eps = eps
    self.min_samples = min_samples
    self.hulls = {0: [], 1: []}

  def fit(self, X, y):
    self.hulls = {0: [], 1: []}
    for class_label in [0, 1]:
      class_points = X[y == class_label]
      dbscan = DBSCAN(eps=self.eps, min_samples=self.min_samples)
      labels = dbscan.fit_predict(class_points)
      unique_labels = set(labels)

      for label in unique_labels:
        if label == -1: continue
        cluster_points = class_points[labels == label]
        if len(cluster_points) >= X.shape[1] + 1:
          try:
            hull = ConvexHull(cluster_points)
            delaunay = Delaunay(hull.points[hull.vertices])
            self.hulls[class_label].append({
                'delaunay': delaunay, 'vertices': hull.points[hull.vertices]
            })
          except Exception as e:
            pass

  def predict(self, X):
    predictions = np.zeros(X.shape[0])
    for i, point in enumerate(X):
      inside_class = None
      for class_label in [0, 1]:
        for hull_info in self.hulls[class_label]:
          if hull_info['delaunay'].find_simplex(point) >= 0:
            inside_class = class_label
            break
        if inside_class is not None: break

      if inside_class is not None:
        predictions[i] = inside_class
      else:
        min_dist = float('inf')
        best_class = 1
        for class_label in [0, 1]:
          for hull_info in self.hulls[class_label]:
            dists = np.linalg.norm(hull_info['vertices'] - point, axis=1)
            closest_vertex_dist = np.min(dists)
            if closest_vertex_dist < min_dist:
              min_dist = closest_vertex_dist
              best_class = class_label
        predictions[i] = best_class
    return predictions

In [ ]:
def evaluate_model(model, name, X_test, y_test):
  y_pred = model.predict(X_test)
  print(f"--- {name} ---")
  print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
  print(f"Precision: {precision_score(y_test, y_pred):.4f}")
  print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
  print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")
  print(f"ROC-AUC:   {roc_auc_score(y_test, y_pred):.4f}\n")

# Instantiate and Train Models
cluster_ch = ClusterCHClassifier(eps=0.5, min_samples=5)
cluster_ch.fit(X_train_pca, y_train)

nearest_ch = NearestClusterCHClassifier(eps=0.5, min_samples=5)
nearest_ch.fit(X_train_pca, y_train)

# Evaluate
evaluate_model(cluster_ch, "Standard Cluster CH Classifier", X_test_pca, y_test)
evaluate_model(nearest_ch, "Nearest Cluster CH Classifier", X_test_pca, y_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Instantiate traditional models
lr_model = LogisticRegression(random_state=random_seed)
rf_model = RandomForestClassifier(random_state=random_seed)
svm_model = SVC(random_state=random_seed)

# Train on the 3D PCA data
lr_model.fit(X_train_pca, y_train)
rf_model.fit(X_train_pca, y_train)
svm_model.fit(X_train_pca, y_train)

# Evaluate traditional models on PCA data
print("=== Traditional Models (3D PCA Data) ===")
evaluate_model(lr_model, "Logistic Regression (PCA)", X_test_pca, y_test)
evaluate_model(rf_model, "Random Forest (PCA)", X_test_pca, y_test)
evaluate_model(svm_model, "SVM Classifier (PCA)", X_test_pca, y_test)

# For context, let's also see what we get with the full embeddings
print("=== Baseline Model (Full Embeddings) ===")
lr_full = LogisticRegression(random_state=random_seed, max_iter=1000)
lr_full.fit(X_train, y_train)
evaluate_model(lr_full, "Logistic Regression (Full Embeddings)", X_test, y_test)

In [ ]:
!pip install --quiet umap-learn

In [ ]:
import umap.umap_ as umap

print("Fitting UMAP (this might take a moment)...")
# Dimensionality Reduction (UMAP)
umap_reducer = umap.UMAP(n_components=3, random_state=random_seed)
X_train_umap = umap_reducer.fit_transform(X_train)
X_test_umap = umap_reducer.transform(X_test)
print("UMAP Transformation Complete!")

In [ ]:
# Re-Train Custom Models on UMAP data
print("=== Custom Models (3D UMAP Data) ===")
cluster_ch_umap = ClusterCHClassifier(eps=0.5, min_samples=5)
cluster_ch_umap.fit(X_train_umap, y_train)

nearest_ch_umap = NearestClusterCHClassifier(eps=0.5, min_samples=5)
nearest_ch_umap.fit(X_train_umap, y_train)

evaluate_model(cluster_ch_umap, "Standard Cluster CH Classifier (UMAP)", X_test_umap, y_test)
evaluate_model(nearest_ch_umap, "Nearest Cluster CH Classifier (UMAP)", X_test_umap, y_test)

# Re-Train Traditional Models on UMAP data
print("=== Traditional Models (3D UMAP Data) ===")
lr_umap = LogisticRegression(random_state=random_seed)
rf_umap = RandomForestClassifier(random_state=random_seed)
svm_umap = SVC(random_state=random_seed)

lr_umap.fit(X_train_umap, y_train)
rf_umap.fit(X_train_umap, y_train)
svm_umap.fit(X_train_umap, y_train)

evaluate_model(lr_umap, "Logistic Regression (UMAP)", X_test_umap, y_test)
evaluate_model(rf_umap, "Random Forest (UMAP)", X_test_umap, y_test)
evaluate_model(svm_umap, "SVM Classifier (UMAP)", X_test_umap, y_test)